In [5]:
from pymongo import MongoClient 
# 1. Conectar al contenedor de Mongo 
client = MongoClient('mongodb',27017) 
# 2. Crear (o usar) la base de datos y la colección 
db = client['MiTulaPrueba'] 
coleccion = db['Scrap_Constanza'] 
# Cambiar por el de cada uno 
print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [6]:
# ==============================================================================
# Módulo de Extracción Profunda: Mitula (Arriendos)
# BLOQUE 2: SCRAPING CON LÍMITE DE 500 REGISTROS
# ==============================================================================

import os
import time
import re
import base64
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --- CONFIGURACIÓN ---
PAGINA_INICIO = 1
PAGINA_FIN = 21
MAX_REGISTROS = 500
# --------------------

os.environ["DISPLAY"] = ":99"
os.system("pkill -9 chrome && pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.* && rm -rf /tmp/.org.chromium.Chromium.*")

print(f"Entorno virtual listo. Asignación Mitula: Páginas {PAGINA_INICIO} a {PAGINA_FIN}")
print(f"Límite máximo de registros: {MAX_REGISTROS}")

options = Options()
# options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0"
)

catalogo_urls = []
datos_finales = []
driver = None

try:
    driver = webdriver.Chrome(options=options)

    url_base = "https://casas.mitula.cl/find?operationType=rent&propertyType=apartment&geoId=R231672"

    # ==============================================================================
    # FASE 1: RECOLECCIÓN EN CATÁLOGO
    # ==============================================================================

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):

        if len(catalogo_urls) >= MAX_REGISTROS:
            break

        print(f"\n--- [FASE 1] Procesando Página {pagina_actual} ---")

        url_pagina = f"{url_base}&page={pagina_actual}"
        driver.get(url_pagina)

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "article.listing-card")
                )
            )

            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)

            time.sleep(1)

            bloques = driver.find_elements(By.CSS_SELECTOR, "article.listing-card")

            for bloque in bloques:

                if len(catalogo_urls) >= MAX_REGISTROS:
                    print(f"\nSe alcanzó el límite de {MAX_REGISTROS} registros.")
                    break

                try:
                    texto_tarjeta = bloque.get_attribute("textContent").lower()

                    # UBICACIÓN
                    if "coquimbo" in texto_tarjeta:
                        ubicacion = "Coquimbo"

                    elif "la serena" in texto_tarjeta:
                        ubicacion = "La Serena"

                    else:
                        continue

                    # URL REAL
                    try:
                        codigo_secreto = bloque.get_attribute("data-clickdestination")

                        if codigo_secreto:
                            url_decodificada = base64.b64decode(
                                codigo_secreto
                            ).decode("utf-8", errors="ignore")

                            url = "https://casas.mitula.cl" + url_decodificada
                        else:
                            continue

                    except:
                        continue

                    if not url or "javascript" in url:
                        continue

                    # PRECIO
                    try:
                        precio_texto = bloque.find_element(
                            By.CSS_SELECTOR,
                            ".price__actual"
                        ).get_attribute("textContent")

                        v_limpio = (
                            precio_texto
                            .replace("$", "")
                            .replace("/mes", "")
                            .replace(".", "")
                            .replace(",", "")
                            .strip()
                        )

                        precio = float(v_limpio) if v_limpio.isdigit() else 0.0

                    except:
                        precio = 0.0

                    if precio == 0.0:
                        continue

                    # DATOS NUMÉRICOS
                    m2 = 0
                    dormitorios = 0
                    banos = 0

                    match_m2 = re.search(r'(\d+)\s*(m2|m²|mts)', texto_tarjeta)
                    if match_m2:
                        m2 = int(match_m2.group(1))

                    match_dorm = re.search(r'(\d+)\s*dormitorio', texto_tarjeta)
                    if match_dorm:
                        dormitorios = int(match_dorm.group(1))

                    match_bano = re.search(r'(\d+)\s*baño', texto_tarjeta)
                    if match_bano:
                        banos = int(match_bano.group(1))

                    # GUARDADO BASE
                    catalogo_urls.append({
                        "responsable": "Constanza Torres",
                        "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "titulo": "Propiedad Mitula",
                        "ubicacion": ubicacion,
                        "m2": m2,
                        "precio": precio,
                        "dormitorios": dormitorios,
                        "baños": banos,
                        "url": url
                    })

                except:
                    continue

            print(f"Registros acumulados: {len(catalogo_urls)}")

        except TimeoutException:
            print("Advertencia: Mitula tardó en cargar")
            continue

    # eliminar duplicados por URL
    catalogo_urls = list(
        {v["url"]: v for v in catalogo_urls}.values()
    )

    print(f"\nCatálogo final: {len(catalogo_urls)} propiedades encontradas")

    # ==============================================================================
    # FASE 2: INSPECCIÓN PROFUNDA
    # ==============================================================================

    for i, propiedad in enumerate(catalogo_urls):

        if (i + 1) % 5 == 0:
            print(f"Progreso FASE 2: {i + 1} de {len(catalogo_urls)}")

        propiedad["estacionamiento"] = 0
        propiedad["piscina"] = 0
        propiedad["quincho"] = 0
        propiedad["terraza"] = 0
        propiedad["gimnasio"] = 0
        propiedad["lavanderia"] = 0

        try:
            driver.get(propiedad["url"])

            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            time.sleep(2)

            texto_total = driver.find_element(
                By.TAG_NAME,
                "body"
            ).get_attribute("textContent").lower()

            if "estacionamiento" in texto_total:
                propiedad["estacionamiento"] = 1

            if "piscina" in texto_total:
                propiedad["piscina"] = 1

            if "quincho" in texto_total:
                propiedad["quincho"] = 1

            if "terraza" in texto_total:
                propiedad["terraza"] = 1

            if "gimnasio" in texto_total:
                propiedad["gimnasio"] = 1

            if "lavander" in texto_total:
                propiedad["lavanderia"] = 1

        except:
            pass

        datos_finales.append(propiedad)

except Exception as e:
    print(f"Error crítico en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()

print(f"\nExtracción finalizada. Total final: {len(datos_finales)} propiedades.")

# ==============================================================================
# GUARDADO EN MONGODB
# ==============================================================================

print("\nGuardando datos en MongoDB...")

for d in datos_finales:
    coleccion.update_one(
        {"url": d["url"]},
        {"$set": d},
        upsert=True
    )

print("Guardado completado")

Entorno virtual listo. Asignación Mitula: Páginas 1 a 21
Límite máximo de registros: 500

--- [FASE 1] Procesando Página 1 ---
Registros acumulados: 29

--- [FASE 1] Procesando Página 2 ---
Registros acumulados: 58

--- [FASE 1] Procesando Página 3 ---
Registros acumulados: 88

--- [FASE 1] Procesando Página 4 ---
Registros acumulados: 116

--- [FASE 1] Procesando Página 5 ---
Registros acumulados: 145

--- [FASE 1] Procesando Página 6 ---
Registros acumulados: 175

--- [FASE 1] Procesando Página 7 ---
Registros acumulados: 204

--- [FASE 1] Procesando Página 8 ---
Registros acumulados: 234

--- [FASE 1] Procesando Página 9 ---
Registros acumulados: 264

--- [FASE 1] Procesando Página 10 ---
Registros acumulados: 294

--- [FASE 1] Procesando Página 11 ---
Registros acumulados: 324

--- [FASE 1] Procesando Página 12 ---
Registros acumulados: 354

--- [FASE 1] Procesando Página 13 ---
Registros acumulados: 383

--- [FASE 1] Procesando Página 14 ---
Registros acumulados: 413

--- [FASE 1]

In [4]:
# ==============================================================================
# VERIFICAR SI LOS DATOS SE GUARDARON BIEN EN MONGODB
# ==============================================================================

from pymongo import MongoClient

try:
    # MISMA conexión que sí funcionó antes
    client = MongoClient('mongodb', 27017)

    # misma BD y colección
    db = client["MiTulaPrueba"]
    coleccion = db["Scrap_Constanza"]

    print("Conexión exitosa a MongoDB")

    # contar registros
    total = coleccion.count_documents({})

    print(f"\nTotal de registros guardados: {total}")

    if total > 0:
        print("\n--- Primeros 5 registros guardados ---\n")

        for doc in coleccion.find().limit(5):
            print(doc)
            print("-" * 80)
    else:
        print("\nLa colección está vacía (no se guardó nada todavía).")

except Exception as e:
    print("Error de conexión o lectura:", e)

Conexión exitosa a MongoDB

Total de registros guardados: 29

--- Primeros 5 registros guardados ---

{'_id': ObjectId('69eef28af182f17a9a4cf621'), 'url': 'https://casas.mitula.cl/adform/24301-256-7ef3-7c60533411b6-814a-199731a-5fb8?page=1&pos=0&t_sec=1&t_pvid=3c3b834e-373a-457a-9bf7-81e295eea209&hd=false', 'baños': 2, 'dormitorios': 3, 'estacionamiento': 1, 'fecha_extraccion': '2026-04-27 05:20:31', 'gimnasio': 0, 'lavanderia': 0, 'm2': 82, 'piscina': 1, 'precio': 600000.0, 'quincho': 0, 'responsable': 'Constanza Torres', 'terraza': 1, 'titulo': 'Propiedad Mitula', 'ubicacion': 'La Serena'}
--------------------------------------------------------------------------------
{'_id': ObjectId('69eef28af182f17a9a4cf623'), 'url': 'https://casas.mitula.cl/adform/24301-256-7c96-8b149a882f4a-a555-19be470-d043?page=1&pos=1&t_sec=1&t_pvid=3c3b834e-373a-457a-9bf7-81e295eea209&hd=false', 'baños': 2, 'dormitorios': 3, 'estacionamiento': 1, 'fecha_extraccion': '2026-04-27 05:20:31', 'gimnasio': 0, 'la

In [10]:
# ==============================================================================
# SOLUCIÓN DEFINITIVA
# Tu Docker NO tiene instalado el conector Mongo-Spark correctamente
# Entonces usaremos PyMongo + Pandas + Spark DataFrame
# (esto siempre funciona)
# ==============================================================================

from pymongo import MongoClient
import pandas as pd
from pyspark.sql import SparkSession

# ------------------------------------------------------------------------------
# 1. CONEXIÓN A MONGODB CON PYMONGO
# ------------------------------------------------------------------------------

client = MongoClient("mongodb://mongodb:27017/")

db = client["MiTulaPrueba"]
coleccion = db["Scrap_Constanza"]

# traer todos los datos
datos = list(coleccion.find())

print("Total registros encontrados en Mongo:", len(datos))

# ------------------------------------------------------------------------------
# 2. CONVERTIR A PANDAS
# ------------------------------------------------------------------------------

df_pandas = pd.DataFrame(datos)

# eliminar _id porque Spark da problemas con ObjectId
if "_id" in df_pandas.columns:
    df_pandas = df_pandas.drop("_id", axis=1)

print("Columnas disponibles:")
print(df_pandas.columns)

# ver primeros registros
print("\nPrimeros registros:")
print(df_pandas.head())

# ------------------------------------------------------------------------------
# 3. CREAR SPARK SESSION NORMAL
# ------------------------------------------------------------------------------

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Analisis_Mitula_Final") \
    .getOrCreate()

print("\nSpark iniciado correctamente")

# ------------------------------------------------------------------------------
# 4. CONVERTIR PANDAS → SPARK DATAFRAME
# ------------------------------------------------------------------------------

df = spark.createDataFrame(df_pandas)

print("\nDataFrame Spark creado correctamente")
print("Total registros:", df.count())

# estructura
df.printSchema()

# mostrar datos
df.show(10, truncate=False)

Total registros encontrados en Mongo: 529
Columnas disponibles:
Index(['url', 'baños', 'dormitorios', 'estacionamiento', 'fecha_extraccion',
       'gimnasio', 'lavanderia', 'm2', 'piscina', 'precio', 'quincho',
       'responsable', 'terraza', 'titulo', 'ubicacion'],
      dtype='object')

Primeros registros:
                                                 url  baños  dormitorios  \
0  https://casas.mitula.cl/adform/24301-256-7ef3-...      2            3   
1  https://casas.mitula.cl/adform/24301-256-7c96-...      2            3   
2  https://casas.mitula.cl/adform/24301-256-77ef-...      1            2   
3  https://casas.mitula.cl/adform/24301-256-786c-...      2            3   
4  https://casas.mitula.cl/adform/24301-256-7b13-...      1            1   

   estacionamiento     fecha_extraccion  gimnasio  lavanderia  m2  piscina  \
0                1  2026-04-27 05:20:31         0           0  82        1   
1                1  2026-04-27 05:20:31         0           0  82        1 

In [11]:
# ==============================================================================
# ANÁLISIS 1: PRECIO PROMEDIO POR UBICACIÓN
# ==============================================================================

from pyspark.sql.functions import avg, round, count

print("----- PRECIO PROMEDIO POR UBICACIÓN -----")

df.groupBy("ubicacion").agg(
    count("*").alias("Cantidad_Propiedades"),
    round(avg("precio"), 0).alias("Precio_Promedio"),
    round(avg("m2"), 0).alias("M2_Promedio")
).orderBy("Precio_Promedio", ascending=False).show()

----- PRECIO PROMEDIO POR UBICACIÓN -----
+---------+--------------------+---------------+-----------+
|ubicacion|Cantidad_Propiedades|Precio_Promedio|M2_Promedio|
+---------+--------------------+---------------+-----------+
|La Serena|                 305|      2010340.0|       62.0|
| Coquimbo|                 224|       522344.0|       61.0|
+---------+--------------------+---------------+-----------+



In [12]:
# ==============================================================================
# ANÁLISIS 2: IMPACTO DE AMENIDADES POR UBICACIÓN
# ==============================================================================

from pyspark.sql.functions import count, avg, round

print("----- IMPACTO DE AMENIDADES POR UBICACIÓN -----")

amenidades = [
    "piscina",
    "quincho",
    "gimnasio",
    "estacionamiento",
    "terraza",
    "lavanderia"
]

for am in amenidades:

    print(f"\n### Análisis de: {am.upper()} ###")

    df.groupBy("ubicacion", am).agg(
        count("*").alias("Cantidad"),
        round(avg("precio"), 0).alias("Precio_Promedio"),
        round(avg("m2"), 0).alias("M2_Promedio")
    ).orderBy("ubicacion").show()

----- IMPACTO DE AMENIDADES POR UBICACIÓN -----

### Análisis de: PISCINA ###
+---------+-------+--------+---------------+-----------+
|ubicacion|piscina|Cantidad|Precio_Promedio|M2_Promedio|
+---------+-------+--------+---------------+-----------+
| Coquimbo|      0|     205|       508610.0|       60.0|
| Coquimbo|      1|      19|       670526.0|       64.0|
|La Serena|      0|     270|      2188273.0|       61.0|
|La Serena|      1|      35|       637714.0|       68.0|
+---------+-------+--------+---------------+-----------+


### Análisis de: QUINCHO ###
+---------+-------+--------+---------------+-----------+
|ubicacion|quincho|Cantidad|Precio_Promedio|M2_Promedio|
+---------+-------+--------+---------------+-----------+
| Coquimbo|      0|     220|       516841.0|       60.0|
| Coquimbo|      1|       4|       825000.0|       73.0|
|La Serena|      0|     285|      2113241.0|       62.0|
|La Serena|      1|      20|       544000.0|       58.0|
+---------+-------+--------+--------

In [13]:
# ==============================================================================
# ANÁLISIS 3: RELACIÓN DORMITORIOS vs PRECIO
# ==============================================================================

print("----- DORMITORIOS VS PRECIO -----")

df.groupBy("ubicacion", "dormitorios").agg(
    count("*").alias("Cantidad"),
    round(avg("precio"), 0).alias("Precio_Promedio")
).orderBy("ubicacion", "dormitorios").show()

----- DORMITORIOS VS PRECIO -----
+---------+-----------+--------+---------------+
|ubicacion|dormitorios|Cantidad|Precio_Promedio|
+---------+-----------+--------+---------------+
| Coquimbo|          1|      13|       420385.0|
| Coquimbo|          2|     118|       492797.0|
| Coquimbo|          3|      85|       578647.0|
| Coquimbo|          4|       7|       529286.0|
| Coquimbo|          5|       1|       500000.0|
|La Serena|          1|      29|       456724.0|
|La Serena|          2|     161|       516762.0|
|La Serena|          3|     112|      4594599.0|
|La Serena|          4|       2|       567500.0|
|La Serena|          5|       1|       980000.0|
+---------+-----------+--------+---------------+



In [14]:
# ==============================================================================
# ANÁLISIS 4: TOP 10 PROPIEDADES MÁS CARAS
# ==============================================================================

print("----- TOP 10 PROPIEDADES MÁS CARAS -----")

df.select(
    "titulo",
    "ubicacion",
    "precio",
    "m2",
    "dormitorios",
    "baños",
    "url"
).orderBy("precio", ascending=False).show(10, truncate=False)

----- TOP 10 PROPIEDADES MÁS CARAS -----
+----------------+---------+---------+---+-----------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|titulo          |ubicacion|precio   |m2 |dormitorios|baños|url                                                                                                                                                     |
+----------------+---------+---------+---+-----------+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|Propiedad Mitula|La Serena|4.5E8    |54 |3          |2    |https://casas.mitula.cl/adform/24301-256-7724-5fb1529f8085-a2c7-19cc9d1-337e?page=6&pos=5&t_sec=1&t_pvid=c0579d18-d9cd-457c-ac6b-5c8f349b4436&hd=false  |
|Propiedad Mitula|La Serena|1500000.0|140|3          |5    |https://casas.mitula.cl/adform/24301-256-7a